In [ ]:
# Solutions live in solutions/. Run from the repository root so imports and paths work.
import os, pathlib, sys
_root = next(p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents] if (p / "workshop").is_dir())
os.chdir(_root)
sys.path.insert(0, str(_root))

# Module 2: An interactive machine-learning playground

**Time:** 40 minutes

You will turn a static model-fitting script into an interactive tool, in four
steps:

1. `interact`: one line, instant UI
2. explicit widgets with `observe` and `Output`
3. a layout with a **Train** button for slow models
4. **Exercise:** a model selector that shows only the right sliders

The data: two interleaving half-moons. Circles are training points, triangles
are test points. The colours show what the model predicts.

In [ ]:
import time

import matplotlib.pyplot as plt
import numpy as np
import ipywidgets as widgets
from IPython.display import display
from sklearn.datasets import make_moons
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC

X, y = make_moons(n_samples=400, noise=0.3, random_state=0)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=0)

GRID_X, GRID_Y = np.meshgrid(np.linspace(-2, 3, 200), np.linspace(-1.5, 2, 200))


def plot_boundary(model, title=""):
    """Plot the model's decision surface with train (o) and test (^) points."""
    grid = np.c_[GRID_X.ravel(), GRID_Y.ravel()]
    if hasattr(model, "predict_proba"):
        z = model.predict_proba(grid)[:, 1]
    else:
        z = 1 / (1 + np.exp(-model.decision_function(grid)))
    fig, ax = plt.subplots(figsize=(5, 4))
    ax.contourf(GRID_X, GRID_Y, z.reshape(GRID_X.shape), levels=20, cmap="RdBu", alpha=0.6, vmin=0, vmax=1)
    ax.scatter(*X_train.T, c=y_train, cmap="RdBu", edgecolor="k", s=20)
    ax.scatter(*X_test.T, c=y_test, cmap="RdBu", edgecolor="k", marker="^", s=35)
    ax.set_title(title)
    ax.set_xticks([])
    ax.set_yticks([])
    plt.show()


def score_text(model):
    return f"train {model.score(X_train, y_train):.3f}   test {model.score(X_test, y_test):.3f}"


print(f"{len(X_train)} training points, {len(X_test)} test points")

## Step 1: `interact` (5 minutes)

`interact` builds sliders from the function's arguments.
`FloatLogSlider` is useful for hyperparameters that span orders of magnitude.

**Try:** make `gamma` very large. What happens to the train and test scores?

In [ ]:
@widgets.interact(
    C=widgets.FloatLogSlider(value=1, base=10, min=-2, max=3),
    gamma=widgets.FloatLogSlider(value=1, base=10, min=-2, max=2),
)
def svm_playground(C, gamma):
    model = SVC(C=C, gamma=gamma).fit(X_train, y_train)
    plot_boundary(model, score_text(model))

Large `gamma` makes the boundary wrap around single points: the train score
goes up and the test score goes down. That is **overfitting**, and you just
*saw* it.

## Step 2: explicit widgets, `observe`, and `Output` (10 minutes)

`interact` is quick but rigid. With explicit widgets you control everything.
`continuous_update=False` means "fit only when the user releases the slider",
which keeps slow code responsive.

In [ ]:
C_slider = widgets.FloatLogSlider(value=1, base=10, min=-2, max=3, description="C",
                                  continuous_update=False)
gamma_slider = widgets.FloatLogSlider(value=1, base=10, min=-2, max=2, description="gamma",
                                      continuous_update=False)
plot_out = widgets.Output()


def update(change=None):
    model = SVC(C=C_slider.value, gamma=gamma_slider.value).fit(X_train, y_train)
    with plot_out:
        plot_out.clear_output(wait=True)
        plot_boundary(model, score_text(model))


for s in (C_slider, gamma_slider):
    s.observe(update, names="value")

update()
widgets.HBox([widgets.VBox([C_slider, gamma_slider]), plot_out])

## Step 3: a layout with a Train button (10 minutes)

A random forest with hundreds of trees is slow. Re-fitting on every slider
change would freeze the notebook, so we add a **Train** button.
`try/finally` makes sure the button is enabled again even if training fails.

In [ ]:
n_trees = widgets.IntSlider(value=100, min=10, max=500, step=10, description="trees")
depth = widgets.IntSlider(value=5, min=1, max=20, description="max depth")
train_button = widgets.Button(description="Train", button_style="primary", icon="play")
rf_scores = widgets.HTML()
rf_out = widgets.Output()


def train(_=None):
    train_button.disabled = True
    train_button.description = "Training..."
    try:
        start = time.perf_counter()
        model = RandomForestClassifier(n_estimators=n_trees.value, max_depth=depth.value,
                                       random_state=0).fit(X_train, y_train)
        seconds = time.perf_counter() - start
        rf_scores.value = f"<b>{score_text(model)}</b><br><i>trained in {seconds:.2f} s</i>"
        with rf_out:
            rf_out.clear_output(wait=True)
            plot_boundary(model, f"{n_trees.value} trees, depth {depth.value}")
    finally:
        train_button.disabled = False
        train_button.description = "Train"


train_button.on_click(train)
train()

controls = widgets.VBox([widgets.HTML("<b>Random forest</b>"), n_trees, depth, train_button, rf_scores])
widgets.HBox([controls, rf_out])

## Step 4: Exercise (15 minutes): a model selector

Complete the cell below so that:

1. `make_model()` returns an **unfitted** model for the chosen model type,
   using the sliders in that model's box.
2. `show_params()` shows only the box of the chosen model.
   Hint: `box.layout.display = "none"` hides a widget, `""` shows it.
3. Changing the model type or any slider re-trains and re-plots.

In [ ]:
model_choice = widgets.ToggleButtons(options=["SVM", "Random forest", "k-NN"], description="model")

svm_C = widgets.FloatLogSlider(value=1, base=10, min=-2, max=3, description="C", continuous_update=False)
svm_gamma = widgets.FloatLogSlider(value=1, base=10, min=-2, max=2, description="gamma", continuous_update=False)
rf_trees = widgets.IntSlider(value=100, min=10, max=300, step=10, description="trees", continuous_update=False)
rf_depth = widgets.IntSlider(value=5, min=1, max=20, description="max depth", continuous_update=False)
knn_k = widgets.IntSlider(value=15, min=1, max=60, description="neighbours", continuous_update=False)

boxes = {
    "SVM": widgets.VBox([svm_C, svm_gamma]),
    "Random forest": widgets.VBox([rf_trees, rf_depth]),
    "k-NN": widgets.VBox([knn_k]),
}
scores = widgets.HTML()
out = widgets.Output()


def make_model():
    if model_choice.value == "SVM":
        return SVC(C=svm_C.value, gamma=svm_gamma.value)
    if model_choice.value == "Random forest":
        return RandomForestClassifier(n_estimators=rf_trees.value, max_depth=rf_depth.value, random_state=0)
    return KNeighborsClassifier(n_neighbors=knn_k.value)


def show_params(change=None):
    for name, box in boxes.items():
        box.layout.display = "" if name == model_choice.value else "none"


def run(change=None):
    model = make_model()
    if model is None:
        scores.value = "<i>make_model() returned None: finish the TODOs</i>"
        return
    model.fit(X_train, y_train)
    scores.value = f"<b>{score_text(model)}</b>"
    with out:
        out.clear_output(wait=True)
        plot_boundary(model, model_choice.value)


model_choice.observe(show_params, names="value")
model_choice.observe(run, names="value")
for slider in (svm_C, svm_gamma, rf_trees, rf_depth, knn_k):
    slider.observe(run, names="value")

show_params()
run()
widgets.HBox([widgets.VBox([model_choice, *boxes.values(), scores]), out])

### Stretch goals

- Add a **noise** slider that regenerates the dataset (`make_moons(noise=...)`).
- Show a confusion matrix next to the plot
  (`sklearn.metrics.ConfusionMatrixDisplay.from_estimator`).
- Add a "Remember best" button that stores the settings with the best test score.

## Key ideas

- `interact` for quick exploration; explicit widgets + `observe` for real tools.
- `continuous_update=False` or a button for slow computations.
- `Output` widgets hold plots and printed text.
- Showing and hiding parts of a UI is just `layout.display`.

**What you cannot do with these controls:** select points *in* the plot.
That needs a custom widget. **Next:** [Module 3, the widget ecosystem](03_widget_ecosystem.ipynb)